# Linear Baseline Training

### Strategy (simple baseline)
- Build a fast baseline for Airbnb price prediction.
- Compare `LinearRegression`, `Ridge`, and `Lasso` on the same split.
- Use dummy variables (`pd.get_dummies`) for categorical features.
- Use `log1p(price)` for training; convert back with `expm1` for metrics.
- Report MAE, RMSE, and R2 to compare models quickly.
- Keep preprocessing minimal to get a clear first benchmark.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [21]:
data_file_path: str = "AB_NYC_2019.csv"

data_frame: pd.DataFrame = pd.read_csv(data_file_path)
print(data_frame.shape)
data_frame.head()

(48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [22]:
# Apply the same core cleaning from EDA.
model_data_frame: pd.DataFrame = data_frame.copy()
model_data_frame["reviews_per_month"] = model_data_frame["reviews_per_month"].fillna(0)
model_data_frame["last_review"] = model_data_frame["last_review"].fillna("No reviews")
model_data_frame = model_data_frame.dropna(subset=["name", "host_name"])
model_data_frame = model_data_frame[model_data_frame["price"] > 0].copy()

# Remove high-end price outliers using the same IQR rule from EDA.
first_quartile: float = model_data_frame["price"].quantile(0.25)
third_quartile: float = model_data_frame["price"].quantile(0.75)
interquartile_range: float = third_quartile - first_quartile
upper_bound: float = third_quartile + 1.5 * interquartile_range
model_data_frame = model_data_frame[model_data_frame["price"] <= upper_bound].copy()

# Drop columns not used by this linear baseline.
columns_to_drop: list[str] = ["id", "name", "host_id", "host_name", "last_review"]
existing_columns_to_drop: list[str] = [column_name for column_name in columns_to_drop if column_name in model_data_frame.columns]
model_data_frame = model_data_frame.drop(columns=existing_columns_to_drop)

# Fill any remaining missing numeric values conservatively.
for column_name in model_data_frame.columns:
    if model_data_frame[column_name].dtype != "object":
        model_data_frame[column_name] = model_data_frame[column_name].fillna(model_data_frame[column_name].median())

# Log-transform target for a stable linear baseline.
target_series: pd.Series = np.log1p(model_data_frame["price"])
feature_data_frame: pd.DataFrame = model_data_frame.drop(columns=["price"])

# One-hot encode categoricals.
feature_data_frame = pd.get_dummies(feature_data_frame, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    feature_data_frame,
    target_series,
    test_size=0.2,
    random_state=42,
)

print(X_train.shape, X_test.shape)
display(X_train.head(5))

(36700, 231) (9176, 231)


,latitude,longitude,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,...,neighbourhood_Whitestone,neighbourhood_Williamsbridge,neighbourhood_Williamsburg,neighbourhood_Willowbrook,neighbourhood_Windsor Terrace,neighbourhood_Woodhaven,neighbourhood_Woodlawn,neighbourhood_Woodside,room_type_Private room,room_type_Shared room
33867,40.67252,-73.93918,2,26,2.21,2,267,True,False,False,...,False,False,False,False,False,False,False,False,False,False
47463,40.65937,-73.93480,1,3,3.00,1,343,True,False,False,...,False,False,False,False,False,False,False,False,True,False
33906,40.69561,-73.94972,30,22,1.83,6,318,True,False,False,...,False,False,False,False,False,False,False,False,True,False
47702,40.72845,-73.95791,2,1,1.00,1,24,True,False,False,...,False,False,False,False,False,False,False,False,False,False
41454,40.70722,-74.01023,2,9,2.11,327,323,False,True,False,...,False,False,False,False,False,False,False,False,False,False


In [23]:
def evaluate_model(model_name: str, actual_log: pd.Series, predicted_log: np.ndarray) -> None:
    actual_price: np.ndarray = np.expm1(actual_log)
    predicted_price: np.ndarray = np.expm1(predicted_log)
    mae_value: float = mean_absolute_error(actual_price, predicted_price)
    rmse_value: float = np.sqrt(mean_squared_error(actual_price, predicted_price))
    r2_value: float = r2_score(actual_price, predicted_price)
    print(f"{model_name}")
    print(f"  MAE:  {mae_value:.2f}")
    print(f"  RMSE: {rmse_value:.2f}")
    print(f"  R2:   {r2_value:.4f}")
    print()

linear_model: LinearRegression = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions: np.ndarray = linear_model.predict(X_test)
evaluate_model("LinearRegression", y_test, linear_predictions)

ridge_model: Ridge = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
ridge_predictions: np.ndarray = ridge_model.predict(X_test)
evaluate_model("Ridge(alpha=1.0)", y_test, ridge_predictions)

lasso_model: Lasso = Lasso(alpha=0.0001, max_iter=10000)
lasso_model.fit(X_train, y_train)
lasso_predictions: np.ndarray = lasso_model.predict(X_test)
evaluate_model("Lasso(alpha=0.001)", y_test, lasso_predictions)


LinearRegression
  MAE:  32.93
  RMSE: 46.93
  R2:   0.5141

Ridge(alpha=1.0)
  MAE:  32.96
  RMSE: 46.97
  R2:   0.5134

Lasso(alpha=0.001)
  MAE:  33.05
  RMSE: 47.08
  R2:   0.5110



Not really a change in hyparparemeter tuning here, R^2 only got worse.
Solid results .5 is okay for a first model. 


### Next steps
- First: tune hyperparameters (`alpha`) for `Ridge` and `Lasso` with cross-validation.
- Example: `GridSearchCV(Ridge(), {"alpha": [0.01, 0.1, 1.0, 10.0]}, cv=5)`.
- Re-run with numeric feature scaling (`StandardScaler`) and compare.
- Add stronger models later (CatBoost/LightGBM/XGBoost) against this baseline.

To further improve predictions and capture more complex relationships, we can train and evaluate:

1. **Random Forest Regressor**  
   A strong ensemble tree model that captures non-linear patterns and feature interactions.

   ```python
   from sklearn.ensemble import RandomForestRegressor

   rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
   rf_model.fit(X_train, y_train)
   rf_predictions = rf_model.predict(X_test)
   evaluate_model("RandomForestRegressor", y_test, rf_predictions)
   ```

2. **Gradient Boosting / XGBoost**  
   A boosting approach that often performs very well on tabular datasets.

   ```python
   from xgboost import XGBRegressor

   xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
   xgb_model.fit(X_train, y_train)
   xgb_predictions = xgb_model.predict(X_test)
   evaluate_model("XGBRegressor", y_test, xgb_predictions)
   ```

Then compare all models by:
- Collecting MAE, RMSE, and R2 in one summary table.
- Visualizing predicted vs actual prices.
- Choosing the best model based on both metrics and behavior.